# Spotify API Dataset Analysis

This notebook analyzes the Spotify API dataset containing track information and audio features.

In [ ]:
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import time
import os
from dotenv import load_dotenv

spotify_df = pd.read_csv('spotify_api.csv', index_col=0)
mirdei_df = pd.read_csv('mirdei_lyrics_cleaned.csv')

print("Spotify dataset shape:", spotify_df.shape)
print("MIRDei lyrics dataset shape:", mirdei_df.shape)

Spotify dataset shape: (114000, 20)
MIRDei lyrics dataset shape: (543, 15)


In [ ]:
load_dotenv()

CLIENT_ID = os.getenv('SPOTIFY_CLIENT_ID')
CLIENT_SECRET = os.getenv('SPOTIFY_CLIENT_SECRET')

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError("Please set SPOTIFY_CLIENT_ID and SPOTIFY_CLIENT_SECRET in your .env file")

sp = SpotifyClientCredentials(client_id=CLIENT_ID, client_secret=CLIENT_SECRET)
spotify = spotipy.Spotify(client_credentials_manager=sp)

In [16]:
def search_track_id(artist, title, spotify_client):

    try:
        query = f"artist:{artist} track:{title}"
        results = spotify_client.search(q=query, type='track', limit=1)
        
        if results['tracks']['items']:
            track = results['tracks']['items'][0]
            return track['id']
        else:
            return None
    except Exception as e:
        print(f"Error searching for {artist} - {title}: {e}")
        return None

track_id = search_track_id("Bob Marley", "Exodus", spotify)
print(f"Track ID: {track_id}")

Track ID: 1lRkDJ9hMuiE1GZOlbgkam


In [ ]:
track_ids = []
total = len(mirdei_df)

for idx, row in mirdei_df.iterrows():
    artist = row['Artist']
    title = row['Title']
    
    track_id = search_track_id(artist, title, spotify)
    track_ids.append(track_id)
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{total} songs... Found {sum(x is not None for x in track_ids)} IDs")
    
    time.sleep(0.5)

mirdei_df['spotify_track_id'] = track_ids

print(f"Found track IDs for {sum(x is not None for x in track_ids)} out of {total} songs")
print(f"Match rate: {sum(x is not None for x in track_ids) / total * 100:.2f}%")

Processed 10/543 songs... Found 10 IDs
Processed 20/543 songs... Found 20 IDs
Processed 20/543 songs... Found 20 IDs
Processed 30/543 songs... Found 30 IDs
Processed 30/543 songs... Found 30 IDs
Processed 40/543 songs... Found 39 IDs
Processed 40/543 songs... Found 39 IDs
Processed 50/543 songs... Found 48 IDs
Processed 50/543 songs... Found 48 IDs
Processed 60/543 songs... Found 57 IDs
Processed 60/543 songs... Found 57 IDs
Processed 70/543 songs... Found 67 IDs
Processed 70/543 songs... Found 67 IDs
Processed 80/543 songs... Found 74 IDs
Processed 80/543 songs... Found 74 IDs
Processed 90/543 songs... Found 83 IDs
Processed 90/543 songs... Found 83 IDs
Processed 100/543 songs... Found 90 IDs
Processed 100/543 songs... Found 90 IDs
Processed 110/543 songs... Found 100 IDs
Processed 110/543 songs... Found 100 IDs
Processed 120/543 songs... Found 110 IDs
Processed 120/543 songs... Found 110 IDs
Processed 130/543 songs... Found 120 IDs
Processed 130/543 songs... Found 120 IDs
Processed 1

In [29]:
# Save dataset with Spotify track IDs for later use
output_df = mirdei_df[['Song', 'Artist', 'Title', 'Quadrant', 'PQuad', 'MoodsTotal', 
                        'Moods', 'MoodsFoundStr', 'MoodsStr', 'MoodsStrSplit', 
                        'Genres', 'GenresStr', 'Sample', 'SampleURL', 'spotify_track_id', 
                        'Lyrics']]

output_df.to_csv('mirdei_with_spotify_ids.csv', index=False)
print(f"\n✓ Saved to 'mirdei_with_spotify_ids.csv'")
print(f"Total songs: {len(output_df)}")
print(f"Songs with Spotify ID: {output_df['spotify_track_id'].notna().sum()}")
print(f"Songs without Spotify ID: {output_df['spotify_track_id'].isna().sum()}")


✓ Saved to 'mirdei_with_spotify_ids.csv'
Total songs: 543
Songs with Spotify ID: 514
Songs without Spotify ID: 29
